# 01 趋势策略研究示例

演示 QIS 平台的研究流程：读取缓存行情 → 换月调整 → 生成权重 → 回测 → 绩效与归因。

前提：已运行 `uv run qis fetch-data` 建好本地缓存。

In [ ]:
import pandas as pd
from qis.data.store import DataStore
from qis.data.universe import Universe
from qis.cli import _adjusted_price_matrix

u = Universe.from_yaml()
store = DataStore()
prices = _adjusted_price_matrix(store, u).loc["2010-01-01":]
print(prices.shape)
prices.tail(3)

In [ ]:
from qis.strategy.trend import trend_weights
from qis.portfolio.construction import vol_target_scale

rets = prices.pct_change(fill_method=None)
w = trend_weights(prices)                       # 策略权重（毛敞口=1）
w = vol_target_scale(w, rets, target=0.10)      # 组合波动目标
w.tail(3)

In [ ]:
from qis.backtest.costs import cost_bps_by_name, load_settings
from qis.backtest.engine import run_backtest
from qis.analytics.metrics import summary

settings = load_settings()
cost = cost_bps_by_name(u.asset_classes(), settings["cost_bps"])
res = run_backtest(prices, w, cost_bps=cost)
summary(res.net, res.turnover).round(3)

In [ ]:
import matplotlib.pyplot as plt
from qis.analytics.report import tearsheet

tearsheet(res.net, res.turnover, title="trend demo")
plt.show()

In [ ]:
# 分标的年化盈亏归因
contrib = ((res.weights * rets.fillna(0.0)).mean() * 252).sort_values()
contrib.plot.barh(figsize=(8, 6), title="Annual P&L contribution by instrument")
plt.show()